In [1]:
import numpy as np
import pyaudio
import wave

TIME = 5
data = []
CHUNK = 1024
RATE = 48000

p = pyaudio.PyAudio()
stream = p.open(format=pyaudio.paInt16, channels=1, rate=RATE, input=True, frames_per_buffer=CHUNK)
w = wave.open("./out.wav", "wb")
w.setnchannels(1)
w.setsampwidth(p.get_sample_size(pyaudio.paInt16))
w.setframerate(RATE)
print("녹음 시작")
try:
    while True:
        w.writeframes(stream.read(CHUNK))
except KeyboardInterrupt:
    pass

w.close()
stream.stop_stream()
stream.close()
p.terminate()

녹음 시작


In [7]:

import wave

import pyaudio

p = pyaudio.PyAudio()
with wave.open("out_loud.wav", "rb") as w:
    data = w.readframes(w.getnframes())
    stream = p.open(format=p.get_format_from_width(2), channels=1, rate=48000, output=True)
    print("재생 시작")
    stream.write(data)     # 블럭킹
    stream.stop_stream()
    print("재생 끝")
    stream.close()

재생 시작
재생 끝


In [2]:
import wave
import numpy as np

with wave.open("out.wav", "rb") as w:
    frames = w.readframes(w.getnframes())
    audio = np.frombuffer(frames, dtype=np.int16)

print("max:", np.max(np.abs(audio)))
print("mean:", np.mean(np.abs(audio)))

max: 4239
mean: 144.4200767263427


In [3]:
import pyaudio
import wave

CHUNK = 1024
RATE = 48000
CHANNELS = 1
FORMAT = pyaudio.paInt16
OUTPUT = "out.wav"

p = pyaudio.PyAudio()

stream = p.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    frames_per_buffer=CHUNK
)

w = wave.open(OUTPUT, "wb")
w.setnchannels(CHANNELS)
w.setsampwidth(p.get_sample_size(FORMAT))
w.setframerate(RATE)

print("녹음 시작 Ctrl+C로 종료")

try:
    while True:
        data = stream.read(CHUNK, exception_on_overflow=False)
        w.writeframes(data)
except KeyboardInterrupt:
    print("녹음 종료")

w.close()
stream.stop_stream()
stream.close()
p.terminate()

녹음 시작 Ctrl+C로 종료
녹음 종료


In [5]:
import wave
import pyaudio
import numpy as np

CHUNK = 1024
RATE = 48000
CHANNELS = 1
FORMAT = pyaudio.paInt16

RAW_WAV = "out.wav"
LOUD_WAV = "out_loud.wav"

GAIN = 5.0


def record_wav(filename):
    p = pyaudio.PyAudio()

    stream = p.open(
        format=FORMAT,
        channels=CHANNELS,
        rate=RATE,
        input=True,
        frames_per_buffer=CHUNK
    )

    w = wave.open(filename, "wb")
    w.setnchannels(CHANNELS)
    w.setsampwidth(p.get_sample_size(FORMAT))
    w.setframerate(RATE)

    print("녹음 시작: Ctrl+C 를 누르면 종료됩니다.")

    try:
        while True:
            data = stream.read(CHUNK, exception_on_overflow=False)
            w.writeframes(data)

    except KeyboardInterrupt:
        print("\n녹음 종료")

    finally:
        w.close()
        stream.stop_stream()
        stream.close()
        p.terminate()


def amplify_wav(input_file, output_file, gain):
    with wave.open(input_file, "rb") as w:
        params = w.getparams()
        frames = w.readframes(w.getnframes())

    audio = np.frombuffer(frames, dtype=np.int16)

    print("원본 최대 진폭:", np.max(np.abs(audio)))

    amplified = audio.astype(np.float32) * gain
    amplified = np.clip(amplified, -32768, 32767)
    amplified = amplified.astype(np.int16)

    print("증폭 후 최대 진폭:", np.max(np.abs(amplified)))

    with wave.open(output_file, "wb") as w:
        w.setparams(params)
        w.writeframes(amplified.tobytes())

    print(f"증폭된 파일 저장 완료: {output_file}")


def main():
    record_wav(RAW_WAV)
    amplify_wav(RAW_WAV, LOUD_WAV, GAIN)


if __name__ == "__main__":
    main()

녹음 시작: Ctrl+C 를 누르면 종료됩니다.

녹음 종료
원본 최대 진폭: 890
증폭 후 최대 진폭: 4450
증폭된 파일 저장 완료: out_loud.wav


In [9]:
# thread 로 구현이 되어 있어서 논블럭하게 쓸 수 있음.
import time
from pop import SoundMeter

sm = SoundMeter()

def onSoundMeter(rms, inData):
    if(rms>600):
        print(rms)

sm.setCallback(onSoundMeter)

input("input something")

sm.stop()

2019
3172
3385
3186
2923
2656
2400
2164
1978
1780
1562
1268
931
619
611
1645


input something asd
